In [25]:
import os
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm
import pickle

# Define path to the dataset (e.g., 'asl_alphabet_train/asl_alphabet_train/A/')
dataset_path = r'C:\Users\hites\Downloads\Python\DeepLearning Projects\Sign Language Detection Model\ASL_Alphabet_Dataset\asl_alphabet_train'
classes = sorted(os.listdir(dataset_path))

# Initialize MediaPipe
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)
data = []
labels = []

# Loop through each class folder and extract landmarks
for label in tqdm(classes):
    folder_path = os.path.join(dataset_path, label)
    for image_file in os.listdir(folder_path)[:300]:  # Limit to 300 images per class
        image_path = os.path.join(folder_path, image_file)
        image = cv2.imread(image_path)
        if image is None:
            continue
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result = hands.process(image_rgb)

        if result.multi_hand_landmarks:
            landmarks = []
            for lm in result.multi_hand_landmarks[0].landmark:
                landmarks.extend([lm.x, lm.y, lm.z])
            data.append(landmarks)
            labels.append(label)

hands.close()

# Save as a pickle file
with open("asl_landmark_data.pkl", "wb") as f:
    pickle.dump((data, labels), f)


  0%|          | 0/26 [00:00<?, ?it/s]

100%|██████████| 26/26 [06:10<00:00, 14.26s/it]


In [26]:
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load landmark data
with open("asl_landmark_data.pkl", "rb") as f:
    data, labels = pickle.load(f)

# Convert to NumPy
X = np.array(data)
y = np.array(labels)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save model
with open("asl_model.pkl", "wb") as f:
    pickle.dump(clf, f)


Accuracy: 0.9439313984168866
              precision    recall  f1-score   support

           A       0.97      0.98      0.98        60
           B       0.97      1.00      0.98        60
           C       1.00      0.96      0.98        56
           D       0.93      0.95      0.94        57
           E       0.93      0.92      0.92        60
           F       0.97      0.98      0.98        60
           G       0.98      0.93      0.96        60
           H       0.95      1.00      0.98        60
           I       0.97      0.95      0.96        59
           J       0.98      0.98      0.98        54
           K       0.95      0.95      0.95        58
           L       0.94      0.98      0.96        60
           M       0.89      0.86      0.88        57
           N       0.87      0.90      0.88        58
           O       0.98      0.95      0.97        59
           P       0.98      0.96      0.97        53
           Q       0.98      0.98      0.98        5